In [ ]:
import pandas as pd
import numpy as np
import os
from nilearn.connectome import ConnectivityMeasure
from nilearn import plotting
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind

# Carregar e limpar os nomes das ROIs (ignorando as 3 primeiras colunas de tecidos globais)
with open('rois_labels.txt', 'r') as f:
    # Remove marcadores de fonte como e linhas vazias
    all_labels = [line.strip() for line in f.readlines() if line.strip() and not line.startswith('[source')]

# Selecionamos apenas as ROIs das redes e atlas (da linha 4 em diante)
roi_names = all_labels[3:] 
print(f"Total de ROIs identificadas para análise: {len(roi_names)}")

In [ ]:
#começar daqui
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from nilearn import plotting
from nilearn.connectome import ConnectivityMeasure
from scipy.stats import ttest_ind

# --- BLOCO 1: CARREGAMENTO E ALINHAMENTO ---
base_path = "./ArquivosNew/GordonEducation/LEiDA_results/time_series"

# Carregar labels e definir ROIs (Ignora os 3 tecidos globais)
with open('rois_labels.txt', 'r') as f:
    roi_names_final = f.readlines()

metadata = pd.read_csv('metadata.csv')
all_timeseries = []
valid_metadata = []

for _, row in metadata.iterrows():
    sub_id = row['subject_id']
    file_path = os.path.join(base_path, sub_id, 'tseries.csv')

    if os.path.exists(file_path):
        df_ts = pd.read_csv(file_path, header=None)
        ts = df_ts.values.T
        all_timeseries.append(ts)
        valid_metadata.append(row)

n_rois = all_timeseries[0].shape[1]

df_final = pd.DataFrame(valid_metadata)
print(f"Sucesso: {len(all_timeseries)} sujeitos carregados com {n_rois} ROIs cada.")

# --- BLOCO 2: CONECTIVIDADE E ESTATÍSTICA ---
conn = ConnectivityMeasure(kind='correlation', vectorize=True, discard_diagonal=True)
stacked_networks = conn.fit_transform(all_timeseries)

# Separar grupos
idx_rel = df_final[df_final['condition'] == 'high'].index
idx_non = df_final[df_final['condition'] == 'low'].index

# Teste-t (Religious vs Non-Religious)
t_stats, p_values = ttest_ind(stacked_networks[idx_rel], stacked_networks[idx_non], axis=0)

# --- BLOCO 3: IDENTIFICAÇÃO DAS CONEXÕES SIGNIFICATIVAS ---
significant_indices = np.where(p_values < 0.05)[0]
triu_idx_map = np.triu_indices(n_rois, k=1)

results = []
for idx in significant_indices:
    roi_a = roi_names_final[triu_idx_map[0][idx]]
    roi_b = roi_names_final[triu_idx_map[1][idx]]
    results.append({'ROI A': roi_a, 'ROI B': roi_b, 'T-stat': t_stats[idx], 'p-value': p_values[idx]})

df_results = pd.DataFrame(results)

if not df_results.empty:
    print("\n### Conexões Significativas (p < 0.05) ###")
    print(df_results.sort_values(by='T-stat', ascending=False).to_string(index=False))
else:
    print("\nNenhuma conexão significativa encontrada com p < 0.05.")

# --- BLOCO 4: MATRIZ DE DIFERENÇAS ---

t_masked = t_stats * (p_values < 0.05)

t_matrix_sig = np.zeros((n_rois, n_rois))
t_matrix_sig[triu_idx_map] = t_masked
t_matrix_sig.T[triu_idx_map] = t_masked

plt.figure(figsize=(12, 10))
plotting.plot_matrix(t_matrix_sig, labels=None, title='Diferenças (T-score)', cmap='RdBu', colorbar=True)
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nilearn import plotting

# --- 1. RECARREGAR COORDENADAS (Caso tenham saído da memória) ---
coords_df = pd.read_csv('rois_coordinates.csv')

coords = coords_df[['X', 'Y', 'Z']].values

# --- 2. CONFIGURAR FILTRO ---
threshold_t = 0.5 

t_matrix_filtered = np.copy(t_matrix_sig)
t_matrix_filtered[np.abs(t_matrix_filtered) < threshold_t] = 0

# --- 3. GERAR O GRÁFICO ---
print(t_matrix_filtered)

plt.figure(figsize=(15, 8))

display = plotting.plot_connectome(
    t_matrix_filtered, 
    coords,
    edge_threshold=threshold_t,
    edge_cmap='RdBu', 
    node_size=10,          
    node_color='black',    
    display_mode='ortho', 
    colorbar=True,
    title=f'Diferenças de Conectividade (|T| > {threshold_t})\nHigh (+) > Low (-)'
)

# Salvar
plt.savefig('mapa_conectividade_limpo.png', dpi=300, bbox_inches='tight')
plt.show()